# Visualize Exported Conversations

This notebook loads all exported conversation JSON files from a directory, shows a summary dataframe with scores, and lets you visualize one conversation by its row index.


In [1]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd()
if not (ROOT / 'verl').exists():
    ROOT = Path('/scratch/ywxzml3j/likaican/src/verl-qwen3-vl')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from verl.utils.vreasoner_v2_conversation_export import (
    load_exported_conversation,
    restore_conversation_for_visualization,
)

# Replace with another export directory if needed.
EXPORT_DIR = Path(
    # "/scratch/ywxzml3j/likaican/temp/iq_ft_eval_json_export_t2/exported_conversations"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/veqa_batch_0350_r2_mveqa_batch_0352_r2-dpi200_aug_noaug_maxp40-resumable"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/iq_base_eval_export_clean_r2/exported_conversations"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/insight_qwen_agent_initial_0_5_default_sys/qwen3-vl-32b-instruct/veqa_batch_0350_r2_mveqa_batch_0352_r2-dpi200_aug_noaug_maxp40/exported_conversations"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/insight_qwen_agent_zoom_factor2_default_sys_resumable/qwen3-vl-32b-instruct/veqa_batch_0350_r2_mveqa_batch_0352_r2-dpi200_aug_noaug_maxp40/exported_conversations"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/insight_qwen_agent_initial_0_5_default_sys/qwen3-vl-32b-instruct/veqa_batch_0350_r2_mveqa_batch_0352_r2-dpi200_aug_noaug_maxp40/exported_conversations"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/O3_data_0424-dpi200_aug_noaug_maxp40-resumable"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/insight_qwen_agent_initial_0_02_zoom_factor2_default_sys_resumable/qwen3-vl-32b-instruct/O3_data_0424-dpi200_aug_noaug_maxp40/exported_conversations"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/O3_data_0424-dpi200_aug_noaug_maxp40-resumable"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/insight_qwen_agent_zoom_factor2_default_sys_0426_resumable/qwen3-vl-32b-instruct/O3_data_0424-dpi200_aug_noaug_maxp40/exported_conversations"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/insight_qwen_agent_initial_0_1_zoom_factor2_default_sys_0426_resumable/qwen3-vl-8b-instruct/O3_data_0424-dpi200_aug_noaug_maxp40/pass0/exported_conversations"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/O3_data_0424-dpi200_aug_noaug_maxp40-0426_resumable"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed_no_rl/O3_data_0424-dpi200_aug_noaug_maxp40-0426_resumable"
    # '/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/insight_qwen_agent_zoom_factor2_default_sys_0426_resumable/qwen3-vl-32b-instruct/veqa_batch_0350_r2_train_mveqa_batch_0352_r2_train-dpi200_aug_noaug_maxp40/train_part1/exported_conversations'
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/veqa_batch_0350_r2_train_mveqa_batch_0352_r2_train-dpi200_aug_noaug_maxp40-0426_train_part1_resumable"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/veqa_batch_0350_r2_train_6508_additional_mveqa_batch_0352_r2_train_6508_additional-dpi200_aug_noaug_maxp40_jitter_seed0_pagedrop0.5_irrel0.3_seed0-0426_train_part3_resumable"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/insight_qwen_agent_full_sft_all_convos_0426_exp1_lr3e-6_cosine_minlr3e-7_len32768_bs32_eval_data_0502_256k_zoom2_area3500/exported_conversations"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/broad_full_clean_stochastic_0503/huggingface_eval_data_0502_256k_zoom2_area3500/exported_conversations"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/broad_full_clean_vs_sys4easy_stochastic_0503/huggingface_eval_data_0502_256k_zoom2_area3500/exported_conversations"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/insight_doc_rl_balanced_dude_reduced_u25_qwen3_insight_qwen_agent_rl_t0_7_def_sparams/dude200_mmlongbench200_o3bench0502_insight_qwen_agent"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/insight_doc_rl_balanced_dude_reduced_u25_qwen3_insight_qwen_agent_rl_t0_7_def_sparams_sft_bs8_epoch_5/dude200_mmlongbench200_o3bench0502_insight_qwen_agent"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/insight_qwen_agent_initial_0_35_default_sys_70dpi_resumable/qwen3-vl-32b-instruct/_medium_processed_wrong_question_manifests_merged_for_parquet_70dpi_part60/exported_conversations"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/_medium_processed_wrong_question_manifests_merged_for_parquet_70dpi_part60-70dpi_resumable"
    # "/home/ywxzml3j/ywxzml3juser40/insight_doc/outputs/insight_qwen_agent_zoom_factor2_area1800_rescale035_default_sys_0426_resumable/qwen3-vl-32b-instruct/veqa_batch_0350_r2_train_6508_additional_mveqa_batch_0352_r2_train_6508_additional-dpi200_aug_noaug_maxp40_jitter_seed0_pagedrop0.5_irrel0.3_seed0/train_part4/exported_conversations"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/veqa_batch_0350_r2_train_6508_additional_mveqa_batch_0352_r2_train_6508_additional-dpi200_aug_noaug_maxp40_jitter_seed0_pagedrop0.5_irrel0.3_seed0-0426_train_part4_resumable"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/veqa_batch_0350_r2_train_6508_additional_mveqa_batch_0352_r2_train_6508_additional-dpi200_aug_noaug_maxp40_jitter_seed0-0426_train_part5_resumable"
    # "/scratch/ywxzml3j/likaican/mms1_rl/exported_conversations/multi_agent_vsearch/synthetic_unanswerable_qwen3_region_loc/verify_all_resumable_az_unanswerable_prompt_verify_gpu4567_explicit_mode3_evidence_fix1"
    "/scratch/ywxzml3j/likaican/src/verl-qwen3-vl/workspace/rl_ckpt700_highpage_standalone_rescale025_035_05_3trials_wc4_overlap_20260608_185422_highpage_wc4_overlap/rescale035_trial0/exported_conversations"
)

export_paths = sorted(EXPORT_DIR.glob("*.json"))
records = [load_exported_conversation(str(p)) for p in export_paths]
len(records)

/home/ywxzml3j/ywxzml3juser40/.conda/envs/vllm-latest/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-09 03:12:05,600	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


367

In [2]:
import pandas as pd

def _get_score_field(record, key, default=None):
    reward = record.get("reward") or {}
    score = reward.get("score") or {}
    return score.get(key, default)

rows = []
for idx, (path, record) in enumerate(zip(export_paths, records)):
    reward = record.get("reward") or {}
    rows.append({
        "idx": idx,
        "subset": record['extra_info']['subset'],
        "file": path.name,
        "job_id": record["job"]["job_id"],
        "agent_name": record.get("agent_name"),
        "question_id": record.get("extra_info", {}).get("question_id"),
        "question": record.get("extra_info", {}).get("question"),
        "critical_failure": record.get("status", {}).get("critical_failure"),
        "reward": reward.get("reward"),
        "score": _get_score_field(record, "score"),
        "format_reward": _get_score_field(record, "format_reward"),
        "accuracy_reward": _get_score_field(record, "accuracy_reward"),
        "n_valid_tool_calls": _get_score_field(record, "n_valid_tool_calls"),
        "extracted_answer": reward.get("extracted_answer"),
        "ground_truth": reward.get("ground_truth"),
        "question_type": record.get("extra_info", {}).get("question_type"),
    })

summary_df = pd.DataFrame(rows)
summary_df


,idx,subset,file,job_id,agent_name,question_id,question,critical_failure,reward,score,format_reward,accuracy_reward,n_valid_tool_calls,extracted_answer,ground_truth,question_type
0,0,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,152c09edd9bc4cd2a5d0136c9c5bc10e,insight_qwen_agent,longdocurl_extract_fig2tab_4039789_38_67_9,List names of the figures at the page which co...,False,0.0,0.0,1.0,0.0,5,Cannot determine,"""[\""F/UTP CAT5E\"", \""SF/UTP CAT5E\"", \""IEC 607...",None
1,1,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,eae0545e2f574f6888b3260f53f162ec,insight_qwen_agent,longdocurl_extract_fig2tab_4057441_40_69_5,What's name of the table at the page which con...,False,1.0,1.0,1.0,1.0,1,Table 17 Power-Up Timing and Write Inhibit Thr...,"""Table 17 Power-Up Timing and Write Inhibit Th...",None
2,2,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,75dcb578400744eabca77653d776e32f,insight_qwen_agent,longdocurl_extract_fig2tab_4110514_44_73_3,List name of the other figure at the page whic...,False,0.0,0.0,1.0,0.0,1,GSS6000,"""[\""GSSEODO\""]""",None
3,3,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,9007d5b2a05941279e020cae66c3a6f9,insight_qwen_agent,longdocurl_extract_fig2tab_4136982_66_88_2,What's name of the figure at the page which co...,False,1.0,1.0,1.0,1.0,2,Ejemplo de una jerarquía,"""Ejemplo de una jerarquia""",None
4,4,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,f714a2f7e75e4f438e3ee357435e3b60,insight_qwen_agent,longdocurl_extract_fig2tab_4149243_19_48_3,List names of the other figures at the page wh...,False,0.0,0.0,1.0,0.0,2,"A-8, A-9, and A-10","""[\""Figure A-7: DMS Case-Ore Mined by Resource...",None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
362,362,Str,mmlongbench0507_highpage-val-trial0-mmlongbenc...,a09f9f246a16428fbb9ed28b783b7837,insight_qwen_agent,mmlongbench_987,What is the most confusing category of Abbrevi...,False,0.0,0.0,1.0,0.0,2,Number,"""Description""",None
363,363,not-answerable,mmlongbench0507_highpage-val-trial0-mmlongbenc...,dd3e09765ae44a48b58d5f90c4202890,insight_qwen_agent,mmlongbench_99,What will happen when you press twice the down...,False,0.0,0.0,1.0,0.0,2,starts a blood pressure measurement,"""Not answerable""",None
364,364,Str,mmlongbench0507_highpage-val-trial0-mmlongbenc...,feb15c1058cf4b2eab9d7522a5f9e6d4,insight_qwen_agent,mmlongbench_992,"At NTU, how many types of Field Sports can stu...",False,1.0,1.0,1.0,1.0,1,four types,"""4""",None
365,365,Str,mmlongbench0507_highpage-val-trial0-mmlongbenc...,95170cfc63d0464ca939cf4c8afd87b8,insight_qwen_agent,mmlongbench_998,How much time does it take from clifton campus...,False,1.0,1.0,1.0,1.0,4,15 mins,"""15 mins""",None


In [3]:
summary_df[summary_df['file'] == 'insight_doc_mixed-val-trial0-dude_ebda6db2b0501934e41cd6c77516a53d_431f2887699ba54a3538f7fa33b87a66-f8d8cde55bee.json']

,idx,subset,file,job_id,agent_name,question_id,question,critical_failure,reward,score,format_reward,accuracy_reward,n_valid_tool_calls,extracted_answer,ground_truth,question_type


In [4]:
summary_df['subset'].value_counts(dropna=False).sort_index()

subset
Float              22
Int                54
List               28
Str                41
answerable        189
not-answerable     33
Name: count, dtype: int64

In [5]:
summary_df[summary_df['accuracy_reward'] == 1.0][summary_df['question_type'] == 'not-answerable']

/tmp/ipykernel_3119940/3622732567.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  summary_df[summary_df['accuracy_reward'] == 1.0][summary_df['question_type'] == 'not-answerable']


,idx,subset,file,job_id,agent_name,question_id,question,critical_failure,reward,score,format_reward,accuracy_reward,n_valid_tool_calls,extracted_answer,ground_truth,question_type


In [6]:
summary_df[summary_df['accuracy_reward'] == 1.0]['subset'].value_counts(dropna=False).sort_index()

subset
Float              10
Int                22
List                7
Str                25
answerable        105
not-answerable     16
Name: count, dtype: int64

In [7]:
summary_df['n_valid_tool_calls'].value_counts(dropna=False).sort_index()

n_valid_tool_calls
0     112
1     106
2      52
3      26
4      24
5      14
6      12
7      14
8       4
9       1
10      2
Name: count, dtype: int64

In [8]:
summary_df[summary_df['reward'] == 1.0]

,idx,subset,file,job_id,agent_name,question_id,question,critical_failure,reward,score,format_reward,accuracy_reward,n_valid_tool_calls,extracted_answer,ground_truth,question_type
1,1,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,eae0545e2f574f6888b3260f53f162ec,insight_qwen_agent,longdocurl_extract_fig2tab_4057441_40_69_5,What's name of the table at the page which con...,False,1.0,1.0,1.0,1.0,1,Table 17 Power-Up Timing and Write Inhibit Thr...,"""Table 17 Power-Up Timing and Write Inhibit Th...",None
3,3,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,9007d5b2a05941279e020cae66c3a6f9,insight_qwen_agent,longdocurl_extract_fig2tab_4136982_66_88_2,What's name of the figure at the page which co...,False,1.0,1.0,1.0,1.0,2,Ejemplo de una jerarquía,"""Ejemplo de una jerarquia""",None
6,6,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,3d99ff9529214caf8047d251d209ecc8,insight_qwen_agent,longdocurl_extract_fig2tab_4175445_48_77_2,What's name of the other figure at the page wh...,False,1.0,1.0,1.0,1.0,2,Figure 4.8,"""Figure 4.8. Light optical cross-section of th...",None
7,7,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,ff4dd9a239134824bed8c824eafe6d70,insight_qwen_agent,longdocurl_extract_fig2tab_4176522_38_67_2,What's name of the figure at the page which co...,False,1.0,1.0,1.0,1.0,1,Figure 3.3,"""Figure 3.3: (left) Saturated atomic absorptio...",None
10,10,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,dfa06ae8e01448dfbcce05c539f332b7,insight_qwen_agent,longdocurl_free_gemini15_pro_4011458_25_54_9,what happens when IDH's mutated?,False,1.0,1.0,1.0,1.0,0,2-hydroxyglutarate accumulation,"""IDH normally converts isocitrate to \u03b1-KG...",None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,359,List,mmlongbench0507_highpage-val-trial0-mmlongbenc...,2a31aa811e1f4b9c9e95e4ccad7af135,insight_qwen_agent,mmlongbench_969,What stages does high level lifecycle have mor...,False,1.0,1.0,1.0,1.0,0,Concept and Production,"""['concept', 'production']""",None
360,360,Str,mmlongbench0507_highpage-val-trial0-mmlongbenc...,46ebe135051b4504af0a6720a983bbd5,insight_qwen_agent,mmlongbench_974,Which baseline did the pre-trained Vicuna-13B ...,False,1.0,1.0,1.0,1.0,2,CoT w. logical constraints baseline,"""CoT w. logical constraints """,None
364,364,Str,mmlongbench0507_highpage-val-trial0-mmlongbenc...,feb15c1058cf4b2eab9d7522a5f9e6d4,insight_qwen_agent,mmlongbench_992,"At NTU, how many types of Field Sports can stu...",False,1.0,1.0,1.0,1.0,1,four types,"""4""",None
365,365,Str,mmlongbench0507_highpage-val-trial0-mmlongbenc...,95170cfc63d0464ca939cf4c8afd87b8,insight_qwen_agent,mmlongbench_998,How much time does it take from clifton campus...,False,1.0,1.0,1.0,1.0,4,15 mins,"""15 mins""",None


In [9]:
summary_df[summary_df['reward'] == 1.0]['n_valid_tool_calls'].value_counts(dropna=False).sort_index()

n_valid_tool_calls
0     68
1     65
2     24
3     10
4      8
5      1
6      3
7      2
8      3
10     1
Name: count, dtype: int64

In [10]:
summary_df[summary_df['reward'] == 1.0][summary_df['n_valid_tool_calls'] > 9]

/tmp/ipykernel_3119940/2999836361.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  summary_df[summary_df['reward'] == 1.0][summary_df['n_valid_tool_calls'] > 9]


,idx,subset,file,job_id,agent_name,question_id,question,critical_failure,reward,score,format_reward,accuracy_reward,n_valid_tool_calls,extracted_answer,ground_truth,question_type
72,72,answerable,longdocurl0507_highpage-val-trial0-longdocurl_...,bc46b185818841338c263f68b7f4acd0,insight_qwen_agent,longdocurl_free_gpt4o_4097628_23_47_4,"In FY 2017, the program received 168 applicati...",False,1.0,1.0,1.0,1.0,10,Yes,"""yes""",None


In [11]:
OMIT_SYSTEM_PROMPT = False

from IPython.display import Markdown, Pretty, display

def display_restored_conversation(restored_payload):
    image_cnt = 0

    def display_next_image():
        nonlocal image_cnt
        image = restored_payload["multi_modal_data"]["images"][image_cnt]
        if image is None:
            display(Pretty("[image unavailable from reference]"))
        else:
            display(image)
            print("image size:", image.size)
        image_cnt += 1

    display(Markdown(f"`[{restored_payload['record']['agent_name']}]`"))

    for message in restored_payload["messages"]:
        display(Markdown(f"**{message['role'].capitalize()}:**"))

        if OMIT_SYSTEM_PROMPT and message["role"] == "system":
            display(Pretty("[system prompt omitted]"))
            continue

        contents = message["content"] if isinstance(message["content"], list) else [message["content"]]
        for content in contents:
            if isinstance(content, str):
                content = {"type": "text", "text": content}

            if content["type"] == "text":
                if content["text"]:
                    display(Pretty(content["text"]))
            elif content["type"] == "image":
                display_next_image()
            else:
                raise ValueError(f"Unknown content type: {content['type']}")

def display_conversation_by_index(idx):
    path = export_paths[idx]
    record = records[idx]
    restored = restore_conversation_for_visualization(record)

    display(summary_df.loc[idx])
    display(pd.json_normalize(record.get("reward") or {}, sep=".").T)
    display(pd.DataFrame([
        {
            "presented_img_idx": item.get("presented_img_idx"),
            "kind": item.get("kind"),
            "source_original_img_idx": item.get("source_original_img_idx"),
            "parent_presented_img_idx": item.get("parent_presented_img_idx"),
            "bbox_on_original": item.get("bbox_on_original"),
            "display_size": item.get("display_size"),
            "region_description": item.get("region_description"),
            "image_restored": item.get("image") is not None,
        }
        for item in restored["presented_images"]
    ]))
    print("EXPORT_PATH:", path)
    display_restored_conversation(restored)


In [12]:
summary_df[summary_df['reward'] == 1.0].index

Index([  1,   3,   6,   7,  10,  11,  14,  15,  16,  18,
       ...
       351, 352, 355, 356, 358, 359, 360, 364, 365, 366],
      dtype='int64', length=185)

In [13]:
summary_df[summary_df['reward'] == 1.0][summary_df['subset'] == 'map'].index[-100:]

/tmp/ipykernel_3119940/3919251105.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  summary_df[summary_df['reward'] == 1.0][summary_df['subset'] == 'map'].index[-100:]


Index([], dtype='int64')

In [14]:
summary_df[summary_df['reward'] == 1.0][summary_df['n_valid_tool_calls'] > 5].index[-100:]

/tmp/ipykernel_3119940/1436475262.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  summary_df[summary_df['reward'] == 1.0][summary_df['n_valid_tool_calls'] > 5].index[-100:]


Index([72, 208, 221, 265, 267, 298, 313, 318, 366], dtype='int64')

In [15]:
summary_df[summary_df['accuracy_reward'] == 1.0][summary_df['question_type'] == 'not-answerable'].index[-100:]

/tmp/ipykernel_3119940/3574248376.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  summary_df[summary_df['accuracy_reward'] == 1.0][summary_df['question_type'] == 'not-answerable'].index[-100:]


Index([], dtype='int64')

In [ ]:
# Pick a row index from summary_df and run this cell.
idx = 0
display_conversation_by_index(idx)
print('extracted answer:', summary_df.loc[idx, 'extracted_answer'])
print('ground truth:', summary_df.loc[idx, 'ground_truth'])